In [14]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression

from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    confusion_matrix, accuracy_score, roc_auc_score, recall_score, f1_score
)

import sklearn 

np.random.seed(42)

df_train = pd.read_csv("train_set.csv")
df_test = pd.read_csv("test_set.csv")
df_train.shape, df_test.shape

((315, 3240), (100, 3240))

In [15]:
X_train_raw = df_train.drop(columns=['ID', 'CLASS']).replace([np.inf, -np.inf], np.nan).replace(np.nan, 0)
y_train = df_train['CLASS']
X_test_raw = df_test.drop(columns=['ID', 'CLASS']).replace([np.inf, -np.inf], np.nan).replace(np.nan, 0)
y_test = df_test['CLASS']

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_raw.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_raw.columns)

### Feature Selection and Training

In [61]:
# === Step 1: Compute Correlations ===
corrs = X_train_scaled_df.corrwith(y_train)  # Pearson correlation
corr_threshold = 0.28
selected_features = corrs[abs(corrs) > corr_threshold].index.tolist()

print(f"Selected {len(selected_features)} features with |correlation| > {corr_threshold}")

Selected 13 features with |correlation| > 0.28


d:\After\torch\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
d:\After\torch\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


In [62]:
y_test.value_counts()

CLASS
0    58
1    42
Name: count, dtype: int64

In [63]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt 


# Train a RandomForestClassifier
model = RandomForestClassifier(n_estimators=1000, random_state=42)
model.fit(X_train_scaled_df, y_train) # Use the DataFrame version for feature names

# Get feature importances
importances = model.feature_importances_

# Create a Series for better visualization
feature_importances = pd.Series(importances, index=X_train_scaled_df.columns).sort_values(ascending=False)

print("\nRandom Forest Feature Importances:")
print(feature_importances)


Random Forest Feature Importances:
Feature_3150    0.002369
Feature_3134    0.002245
Feature_3054    0.002016
Feature_1957    0.002003
Feature_1683    0.001902
                  ...   
Feature_2773    0.000000
Feature_2774    0.000000
Feature_2124    0.000000
Feature_2123    0.000000
Feature_2116    0.000000
Length: 3238, dtype: float64


In [ ]:
y_pred = model.predict(X_test_scaled)
y_pred_proba = model.predict_proba(X_test_scaled)[:, 1] # Probabilities for the positive class

print("\nPredictions made on X_test_scaled.")

print("\n--- Model Evaluation on Test Data ---")

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(sklearn.metrics.classification_report(y_test, y_pred))

roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"ROC AUC Score: {roc_auc:.4f}")


Predictions made on X_test_scaled.

--- Model Evaluation on Test Data ---
Accuracy: 0.6000

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.79      0.70        58
           1       0.54      0.33      0.41        42

    accuracy                           0.60       100
   macro avg       0.58      0.56      0.55       100
weighted avg       0.59      0.60      0.58       100

ROC AUC Score: 0.6755


d:\After\torch\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
d:\After\torch\Lib\site-packages\sklearn\base.py:493: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [65]:
selected_features = feature_importances[:51].keys()

In [66]:
with open('random_forest_features.txt', 'w') as f:
    f.write(','.join(str(feature) for feature in selected_features))

In [67]:
with open('random_forest_features.txt', 'r') as f:
    content = f.read()
    features = content.split(',')
    
features

['Feature_3150',
 'Feature_3134',
 'Feature_3054',
 'Feature_1957',
 'Feature_1683',
 'Feature_1695',
 'Feature_3214',
 'Feature_3070',
 'Feature_2057',
 'Feature_2965',
 'Feature_2726',
 'Feature_2925',
 'Feature_3182',
 'Feature_448',
 'Feature_75',
 'Feature_242',
 'Feature_2233',
 'Feature_1701',
 'Feature_3203',
 'Feature_743',
 'Feature_260',
 'Feature_3139',
 'Feature_1854',
 'Feature_599',
 'Feature_1712',
 'Feature_2720',
 'Feature_2196',
 'Feature_1693',
 'Feature_1757',
 'Feature_1993',
 'Feature_2308',
 'Feature_3166',
 'Feature_1807',
 'Feature_1711',
 'Feature_2324',
 'Feature_1558',
 'Feature_2790',
 'Feature_2109',
 'Feature_2054',
 'Feature_1447',
 'Feature_1431',
 'Feature_1837',
 'Feature_1675',
 'Feature_217',
 'Feature_1030',
 'Feature_223',
 'Feature_2957',
 'Feature_3198',
 'Feature_248',
 'Feature_2356',
 'Feature_423']

In [ ]:
X_train_corr = X_train_raw[features]
X_test_corr = X_test_raw[features]

scaler = StandardScaler()
X_train_corr_scaled = scaler.fit_transform(X_train_corr)
X_test_corr_scaled = scaler.transform(X_test_corr)

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train_corr_scaled, y_train)
y_pred_train = clf.predict(X_train_corr_scaled)
y_pred_test = clf.predict(X_test_corr_scaled)
y_proba_test = clf.predict_proba(X_test_corr_scaled)[:, 1]

cm = sklearn.metrics.confusion_matrix(y_test, y_pred_test)
tn, fp, fn, tp = cm.ravel()
specificity = tn / (tn + fp)  # Specificity = TN / (TN + FP)

print("\n=== Correlation-Based Selection + Logistic Regression ===")
print("Train Accuracy:", accuracy_score(y_train, y_pred_train))
print("Test Accuracy :", accuracy_score(y_test, y_pred_test))
print("Test AUROC    :", roc_auc_score(y_test, y_proba_test))
print("Test Sensitivity (Recall/TPR):", recall_score(y_test, y_pred_test))
print("Test Specificity (TNR)      :", specificity)
print("Test F1-score :", f1_score(y_test, y_pred_test))


=== Correlation-Based Selection + Logistic Regression ===
Train Accuracy: 0.6793650793650794
Test Accuracy : 0.69
Test AUROC    : 0.6732348111658456
Test Sensitivity (Recall/TPR): 0.4523809523809524
Test Specificity (TNR)      : 0.8620689655172413
Test F1-score : 0.5507246376811594


In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

X_train_corr = X_train_raw[features]
X_test_corr = X_test_raw[features]

scaler = StandardScaler()
X_train_corr_scaled = scaler.fit_transform(X_train_corr)
X_test_corr_scaled = scaler.transform(X_test_corr)

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)
    y_proba_test = model.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = sklearn.metrics.confusion_matrix(y_test, y_pred_test).ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    print(f"\n=== {name} ===")
    print("Train Accuracy:", accuracy_score(y_train, y_pred_train))
    print("Test Accuracy :", accuracy_score(y_test, y_pred_test))
    print("Test AUROC    :", roc_auc_score(y_test, y_proba_test))
    print("Test Sensitivity (Recall/TPR):", recall_score(y_test, y_pred_test))
    print("Test Specificity (TNR)      :", specificity)
    print("Test F1-score :", f1_score(y_test, y_pred_test))

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_corr_scaled, y_train)
evaluate_model("Logistic Regression", lr, X_train_corr_scaled, X_test_corr_scaled, y_train, y_test)

svm = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
svm.fit(X_train_corr_scaled, y_train)
evaluate_model("SVM (RBF Kernel)", svm, X_train_corr_scaled, X_test_corr_scaled, y_train, y_test)

rf = RandomForestClassifier(n_estimators=100, max_depth=None, random_state=42)
rf.fit(X_train_corr_scaled, y_train)
evaluate_model("Random Forest", rf, X_train_corr_scaled, X_test_corr_scaled, y_train, y_test)

ensemble = VotingClassifier(estimators=[
    ('lr', lr),
    ('svm', svm),
    ('rf', rf)
], voting='soft') 
ensemble.fit(X_train_corr_scaled, y_train)
evaluate_model("Ensemble (Voting Classifier)", ensemble, X_train_corr_scaled, X_test_corr_scaled, y_train, y_test)


=== Logistic Regression ===
Train Accuracy: 0.6793650793650794
Test Accuracy : 0.69
Test AUROC    : 0.6732348111658456
Test Sensitivity (Recall/TPR): 0.4523809523809524
Test Specificity (TNR)      : 0.8620689655172413
Test F1-score : 0.5507246376811594

=== SVM (RBF Kernel) ===
Train Accuracy: 0.7333333333333333
Test Accuracy : 0.61
Test AUROC    : 0.6633825944170771
Test Sensitivity (Recall/TPR): 0.2619047619047619
Test Specificity (TNR)      : 0.8620689655172413
Test F1-score : 0.36065573770491804

=== Random Forest ===
Train Accuracy: 1.0
Test Accuracy : 0.62
Test AUROC    : 0.6366995073891626
Test Sensitivity (Recall/TPR): 0.35714285714285715
Test Specificity (TNR)      : 0.8103448275862069
Test F1-score : 0.4411764705882353

=== Ensemble (Voting Classifier) ===
Train Accuracy: 0.8920634920634921
Test Accuracy : 0.66
Test AUROC    : 0.6863711001642037
Test Sensitivity (Recall/TPR): 0.38095238095238093
Test Specificity (TNR)      : 0.8620689655172413
Test F1-score : 0.4848484848484

### Overall Search

In [4]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import make_scorer, roc_auc_score, accuracy_score
import numpy as np
from sklearn.metrics import (
    confusion_matrix, accuracy_score, roc_auc_score, recall_score, f1_score
)

models = {
    "LogisticRegression": {
        "model": LogisticRegression(max_iter=1000),
        "params": {
            'model__C': [0.01, 0.1, 1, 10]
        }
    },
    "RandomForest": {
        "model": RandomForestClassifier(random_state=42),
        "params": {
            'model__n_estimators': [50, 100],
            'model__max_depth': [None, 10, 20]
        }
    },
    "GradientBoosting": {
        "model": GradientBoostingClassifier(random_state=42),
        "params": {
            'model__n_estimators': [50, 100],
            'model__learning_rate': [0.01, 0.1]
        }
    },
    "AdaBoost": {
        "model": AdaBoostClassifier(random_state=42),
        "params": {
            'model__n_estimators': [50, 100],
            'model__learning_rate': [0.01, 0.1, 1]
        }
    },
    "ExtraTrees": {
        "model": ExtraTreesClassifier(random_state=42),
        "params": {
            'model__n_estimators': [50, 100],
            'model__max_depth': [None, 10, 20]
        }
    },
    "SVC": {
        "model": SVC(probability=True),
        "params": {
            'model__C': [0.1, 1, 10],
            'model__kernel': ['linear', 'rbf']
        }
    },
    "NaiveBayes": {
        "model": GaussianNB(),
        "params": {}
    },
    "KNN": {
        "model": KNeighborsClassifier(),
        "params": {
            'model__n_neighbors': [3, 5, 7]
        }
    },
    "DecisionTree": {
        "model": DecisionTreeClassifier(random_state=42),
        "params": {
            'model__max_depth': [None, 10, 20],
            'model__min_samples_split': [2, 5]
        }
    }
}

# Scorer
scorer = make_scorer(roc_auc_score, needs_proba=True)

# Results dictionary
results = {}

# StratifiedKFold
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Feature selector for high-dimensional low-sample problems
feature_selector = SelectFromModel(RandomForestClassifier(n_estimators=100, random_state=42), threshold="median")



final_results = {}

for name, model_def in models.items():
    print(f"\n=== {name} ===")
    
    # Refit GridSearchCV to get the best model
    pipe = Pipeline([
        ('feature_selection', feature_selector),
        ('model', model_def["model"])
    ])
    clf = GridSearchCV(pipe, model_def["params"], cv=cv, scoring=scorer, n_jobs=-1)
    clf.fit(X_train_scaled_df, y_train)
    best_model = clf.best_estimator_
    
    # Predictions
    y_pred_train = best_model.predict(X_train_scaled_df)
    y_pred_test = best_model.predict(X_test_scaled_df)
    y_proba_test = best_model.predict_proba(X_test_scaled_df)[:, 1]

    # Metrics
    cm = confusion_matrix(y_test, y_pred_test)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0

    train_acc = accuracy_score(y_train, y_pred_train)
    test_acc = accuracy_score(y_test, y_pred_test)
    test_roc_auc = roc_auc_score(y_test, y_proba_test)
    test_recall = recall_score(y_test, y_pred_test)
    test_f1 = f1_score(y_test, y_pred_test)

    print("Train Accuracy:", train_acc)
    print("Test Accuracy :", test_acc)
    print("Test AUROC    :", test_roc_auc)
    print("Test Sensitivity (Recall/TPR):", test_recall)
    print("Test Specificity (TNR)      :", specificity)
    print("Test F1-score :", test_f1)

    final_results[name] = {
        "best_params": clf.best_params_,
        "train_accuracy": train_acc,
        "test_accuracy": test_acc,
        "test_roc_auc": test_roc_auc,
        "test_sensitivity": test_recall,
        "test_specificity": specificity,
        "test_f1_score": test_f1
    }



=== LogisticRegression ===


d:\After\torch\Lib\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan]
  warnings.warn(


Train Accuracy: 0.8095238095238095
Test Accuracy : 0.66
Test AUROC    : 0.69376026272578
Test Sensitivity (Recall/TPR): 0.42857142857142855
Test Specificity (TNR)      : 0.8275862068965517
Test F1-score : 0.5142857142857142

=== RandomForest ===


d:\After\torch\Lib\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
  warnings.warn(


Train Accuracy: 1.0
Test Accuracy : 0.62
Test AUROC    : 0.6362889983579639
Test Sensitivity (Recall/TPR): 0.2619047619047619
Test Specificity (TNR)      : 0.8793103448275862
Test F1-score : 0.36666666666666664

=== GradientBoosting ===


...

Train Accuracy: 0.5047619047619047
Test Accuracy : 0.5
Test AUROC    : 0.5517241379310345
Test Sensitivity (Recall/TPR): 0.9285714285714286
Test Specificity (TNR)      : 0.1896551724137931
Test F1-score : 0.609375

=== KNN ===


d:\After\torch\Lib\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan]
  warnings.warn(


Train Accuracy: 0.7682539682539683
Test Accuracy : 0.62
Test AUROC    : 0.6674876847290641
Test Sensitivity (Recall/TPR): 0.42857142857142855
Test Specificity (TNR)      : 0.7586206896551724
Test F1-score : 0.4864864864864865

=== DecisionTree ===


d:\After\torch\Lib\site-packages\sklearn\model_selection\_search.py:1108: UserWarning: One or more of the test scores are non-finite: [nan nan nan nan nan nan]
  warnings.warn(


Train Accuracy: 1.0
Test Accuracy : 0.58
Test AUROC    : 0.5656814449917899
Test Sensitivity (Recall/TPR): 0.47619047619047616
Test Specificity (TNR)      : 0.6551724137931034
Test F1-score : 0.4878048780487805


### Features 

In [ ]:
with open('random_forest_features.txt', 'r') as f:
    content = f.read()
    features = content.split(',')

X_train_best = X_train_raw[features]
X_test_best = X_test_raw[features]

# === Step 2: Standardize ===
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_best)
X_test_scaled = scaler.transform(X_test_best)


models = [
    ('lr', LogisticRegression( max_iter=1000, random_state=42)),
    ('rf', RandomForestClassifier(n_estimators=50, max_depth=30, random_state=42)),
    ('svc', SVC(C=2, kernel='rbf', probability=True, random_state=42)),
    ('gb', GradientBoostingClassifier(n_estimators=50, learning_rate=0.01, random_state=42)),
    ('ada', AdaBoostClassifier(n_estimators=1000, learning_rate=0.01, random_state=42)),
    ('et', ExtraTreesClassifier(n_estimators=1000, max_depth=30, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=3)),
    ('nb', GaussianNB()),
    ('dt', DecisionTreeClassifier(max_depth=30, random_state=42))
]

ensemble = VotingClassifier(estimators=models, voting='soft', n_jobs=-1)
ensemble.fit(X_train_scaled, y_train)


y_pred_test = ensemble.predict(X_test_scaled)
y_proba_test = ensemble.predict_proba(X_test_scaled)[:, 1]
y_pred_train = ensemble.predict(X_train_scaled)
y_proba_train = ensemble.predict_proba(X_train_scaled)[:, 1]


def get_specificity(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)


print("\n=== Enhanced Ensemble (Soft Voting, Best Feature Subset) ===")
print("Train Accuracy :", accuracy_score(y_train, y_pred_train))
print("Test Accuracy  :", accuracy_score(y_test, y_pred_test))
print("Test AUROC     :", roc_auc_score(y_test, y_proba_test))
print("Test F1-score  :", f1_score(y_test, y_pred_test))
print("Test Recall (Sensitivity / TPR):", recall_score(y_test, y_pred_test))
print("Test Specificity (TNR)          :", get_specificity(y_test, y_pred_test))


=== Enhanced Ensemble (Soft Voting, Best Feature Subset) ===
Train Accuracy : 0.9396825396825397
Test Accuracy  : 0.6
Test AUROC     : 0.6691297208538588
Test F1-score  : 0.4594594594594595
Test Recall (Sensitivity / TPR): 0.40476190476190477
Test Specificity (TNR)          : 0.7413793103448276


### Only Random Forest and SVM

In [ ]:
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
svm = SVC(C=1, kernel='rbf', probability=True, random_state=42)

ensemble = VotingClassifier(
    estimators=[('rf', rf), ('svm', svm)],
    voting='soft',
    n_jobs=-1
)
ensemble.fit(X_train_scaled, y_train)

y_pred_train = ensemble.predict(X_train_scaled)
y_pred_test = ensemble.predict(X_test_scaled)
y_proba_test = ensemble.predict_proba(X_test_scaled)[:, 1]

def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

print("\n=== Ensemble: RandomForest + SVM ===")
print("Train Accuracy :", accuracy_score(y_train, y_pred_train))
print("Test Accuracy  :", accuracy_score(y_test, y_pred_test))
print("Test AUROC     :", roc_auc_score(y_test, y_proba_test))
print("Test F1-score  :", f1_score(y_test, y_pred_test))
print("Test Recall (Sensitivity / TPR):", recall_score(y_test, y_pred_test))
print("Test Specificity (TNR)          :", specificity_score(y_test, y_pred_test))


=== Ensemble: RandomForest + SVM ===
Train Accuracy : 0.9523809523809523
Test Accuracy  : 0.62
Test AUROC     : 0.6691297208538587
Test F1-score  : 0.3870967741935484
Test Recall (Sensitivity / TPR): 0.2857142857142857
Test Specificity (TNR)          : 0.8620689655172413


In [32]:
def specificity_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    return tn / (tn + fp)

# === Random Forest ===
rf = RandomForestClassifier(n_estimators=300, max_depth=20, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_pred = rf.predict(X_test_scaled)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]

print("\n=== Random Forest Results ===")
print("Accuracy      :", accuracy_score(y_test, rf_pred))
print("AUROC         :", roc_auc_score(y_test, rf_proba))
print("F1-score      :", f1_score(y_test, rf_pred))
print("Recall (TPR)  :", recall_score(y_test, rf_pred))
print("Specificity   :", specificity_score(y_test, rf_pred))

# === SVM ===
svm = SVC(C=30, kernel='rbf', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)
svm_proba = svm.predict_proba(X_test_scaled)[:, 1]

print("\n=== SVM Results ===")
print("Accuracy      :", accuracy_score(y_test, svm_pred))
print("AUROC         :", roc_auc_score(y_test, svm_proba))
print("F1-score      :", f1_score(y_test, svm_pred))
print("Recall (TPR)  :", recall_score(y_test, svm_pred))
print("Specificity   :", specificity_score(y_test, svm_pred))


=== Random Forest Results ===
Accuracy      : 0.59
AUROC         : 0.6379310344827586
F1-score      : 0.4383561643835616
Recall (TPR)  : 0.38095238095238093
Specificity   : 0.7413793103448276

=== SVM Results ===
Accuracy      : 0.72
AUROC         : 0.7249589490968802
F1-score      : 0.6216216216216216
Recall (TPR)  : 0.5476190476190477
Specificity   : 0.8448275862068966


In [ ]:

svm = SVC(C=50, kernel='linear', probability=True, random_state=42)
svm.fit(X_train_scaled, y_train)
svm_pred = svm.predict(X_test_scaled)
svm_proba = svm.predict_proba(X_test_scaled)[:, 1]

print("\n=== SVM Results ===")
print("Accuracy      :", accuracy_score(y_test, svm_pred))
print("AUROC         :", roc_auc_score(y_test, svm_proba))
print("F1-score      :", f1_score(y_test, svm_pred))
print("Recall (TPR)  :", recall_score(y_test, svm_pred))
print("Specificity   :", specificity_score(y_test, svm_pred))


=== SVM Results ===
Accuracy      : 0.69
AUROC         : 0.6463464696223317
F1-score      : 0.5753424657534246
Recall (TPR)  : 0.5
Specificity   : 0.8275862068965517


In [ ]:
# Dictionary of models to train
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(max_depth = 30, learning_rate = 0.05, criterion = 'squared_error', random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Naive Bayes": GaussianNB(),
    "Linear SVM": SVC(kernel='linear', C=1, probability=True, random_state=42),
    "RBF SVM": SVC(kernel='rbf', C=1, probability=True, random_state=42)
}

def evaluate_model(name, y_true, y_pred, y_proba):
    print(f"\n=== {name} Results ===")
    print("Accuracy      :", accuracy_score(y_true, y_pred))
    print("AUROC         :", roc_auc_score(y_true, y_proba))
    print("F1-score      :", f1_score(y_true, y_pred))
    print("Recall (TPR)  :", recall_score(y_true, y_pred))
    print("Specificity   :", specificity_score(y_true, y_pred))

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    
    y_pred = model.predict(X_test_scaled)
    
    try:
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
    except AttributeError:
        if hasattr(model, "decision_function"):
            decision_scores = model.decision_function(X_test_scaled)
            import scipy.special
            y_proba = scipy.special.expit(decision_scores)
        else:
            y_proba = y_pred
    
    # Evaluate
    evaluate_model(name, y_test, y_pred, y_proba)


=== Decision Tree Results ===
Accuracy      : 0.56
AUROC         : 0.5517241379310345
F1-score      : 0.4883720930232558
Recall (TPR)  : 0.5
Specificity   : 0.603448275862069

=== Random Forest Results ===
Accuracy      : 0.62
AUROC         : 0.6366995073891626
F1-score      : 0.4411764705882353
Recall (TPR)  : 0.35714285714285715
Specificity   : 0.8103448275862069

=== Gradient Boosting Results ===
Accuracy      : 0.54
AUROC         : 0.5392036124794746
F1-score      : 0.4523809523809524
Recall (TPR)  : 0.4523809523809524
Specificity   : 0.603448275862069

=== AdaBoost Results ===
Accuracy      : 0.57
AUROC         : 0.5669129720853859
F1-score      : 0.410958904109589
Recall (TPR)  : 0.35714285714285715
Specificity   : 0.7241379310344828

=== Extra Trees Results ===
Accuracy      : 0.6
AUROC         : 0.6678981937602627
F1-score      : 0.42857142857142855
Recall (TPR)  : 0.35714285714285715
Specificity   : 0.7758620689655172

=== K-Nearest Neighbors Results ===
Accuracy      : 0.65


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

gnb = GaussianNB()
param_grid = {
    'var_smoothing': np.logspace(-12, -6, 20)
}

# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(gnb, param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train_scaled, y_train)

# Best model
best_gnb = grid.best_estimator_

y_pred = best_gnb.predict(X_test_scaled)
y_proba = best_gnb.predict_proba(X_test_scaled)[:, 1]

print("\n=== Tuned GaussianNB Results ===")
print("Best var_smoothing:", grid.best_params_['var_smoothing'])
print("Accuracy      :", accuracy_score(y_test, y_pred))
print("AUROC         :", roc_auc_score(y_test, y_proba))
print("F1-score      :", f1_score(y_test, y_pred))
print("Recall (TPR)  :", recall_score(y_test, y_pred))
print("Specificity   :", specificity_score(y_test, y_pred))


=== Tuned GaussianNB Results ===
Best var_smoothing: 1e-12
Accuracy      : 0.63
AUROC         : 0.6564039408866995
F1-score      : 0.5747126436781609
Recall (TPR)  : 0.5952380952380952
Specificity   : 0.6551724137931034


In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

model = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth = 5, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp)

print("\n=== HistGradientBoostingClassifier ===")
print("Accuracy      :", accuracy_score(y_test, y_pred))
print("AUROC         :", roc_auc_score(y_test, y_proba))
print("F1-score      :", f1_score(y_test, y_pred))
print("Recall (TPR)  :", recall_score(y_test, y_pred))
print("Specificity   :", specificity)


=== HistGradientBoostingClassifier ===
Accuracy      : 0.58
AUROC         : 0.6268472906403941
F1-score      : 0.43243243243243246
Recall (TPR)  : 0.38095238095238093
Specificity   : 0.7241379310344828


In [57]:
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis

qda = QuadraticDiscriminantAnalysis()
qda.fit(X_train_scaled, y_train)

y_pred = qda.predict(X_test_scaled)
y_proba = qda.predict_proba(X_test_scaled)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
specificity = tn / (tn + fp)

print("\n=== Quadratic Discriminant Analysis ===")
print("Accuracy      :", accuracy_score(y_test, y_pred))
print("AUROC         :", roc_auc_score(y_test, y_proba))
print("F1-score      :", f1_score(y_test, y_pred))
print("Recall (TPR)  :", recall_score(y_test, y_pred))
print("Specificity   :", specificity)


=== Quadratic Discriminant Analysis ===
Accuracy      : 0.63
AUROC         : 0.6176108374384236
F1-score      : 0.5647058823529412
Recall (TPR)  : 0.5714285714285714
Specificity   : 0.6724137931034483


d:\After\torch\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
d:\After\torch\Lib\site-packages\sklearn\discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


### Further tries

In [59]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier


from sklearn.preprocessing import StandardScaler
import sklearn 
from sklearn.metrics import (
    confusion_matrix, accuracy_score, roc_auc_score, recall_score, f1_score
)
import os 

df_train = pd.read_csv("train_set.csv")
df_test = pd.read_csv("test_set.csv")
blind  = pd.read_csv("blinded_test_set.csv")

X_train_raw = df_train.drop(columns=['ID', 'CLASS']).replace([np.inf, -np.inf], np.nan).replace(np.nan, 0)
y_train = df_train['CLASS']
X_test_raw = df_test.drop(columns=['ID', 'CLASS']).replace([np.inf, -np.inf], np.nan).replace(np.nan, 0)
y_test = df_test['CLASS']

X_blind_raw = blind.drop(columns=["ID"]).replace([np.inf, -np.inf], np.nan).replace(np.nan, 0)
blind_ids = blind["ID"]

scaler = StandardScaler()
X_train_corr_scaled = scaler.fit_transform(X_train_raw)
X_test_corr_scaled = scaler.transform(X_test_raw)
X_blind_corr_scaled = scaler.transform(X_blind_raw)


In [ ]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, f1_score, confusion_matrix, make_scorer
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def evaluate(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    specificity = tn / (tn + fp)
    return acc, auc, recall, specificity, f1

param_grid_enhanced = {
    'n_neighbors': [1, 3, 5, 7, 9, 11, 13, 15, 17, 19, 21], 
    'weights': ['uniform', 'distance'],
    'p': [1, 2],
    'metric': ['minkowski', 'euclidean', 'manhattan', 'chebyshev'], 
    'algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute'], 
}

param_grid_focused = {
    'n_neighbors': [3, 5, 7, 9, 11, 13, 15],
    'weights': ['uniform', 'distance'],
    'p': [1, 2],
    'metric': ['minkowski', 'euclidean', 'manhattan'],
}

def custom_accuracy_scorer(estimator, X, y):
    y_pred = estimator.predict(X)
    return accuracy_score(y, y_pred)

print("=== Method 1: Grid Search optimizing for Accuracy ===")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_acc = GridSearchCV(
    KNeighborsClassifier(), 
    param_grid_focused, 
    scoring='accuracy', 
    cv=cv, 
    n_jobs=-1, 
    verbose=1
)
grid_acc.fit(X_train_corr_scaled, y_train)
best_model_acc = grid_acc.best_estimator_

print(f"Best KNN Params (Accuracy): {grid_acc.best_params_}")
print(f"Best CV Accuracy Score: {grid_acc.best_score_:.4f}")

print("\n=== Method 2: Grid Search optimizing for F1 Score ===")
grid_f1 = GridSearchCV(
    KNeighborsClassifier(), 
    param_grid_focused, 
    scoring='f1',
    cv=cv, 
    n_jobs=-1, 
    verbose=1
)
grid_f1.fit(X_train_corr_scaled, y_train)
best_model_f1 = grid_f1.best_estimator_

print(f"Best KNN Params (F1): {grid_f1.best_params_}")
print(f"Best CV F1 Score: {grid_f1.best_score_:.4f}")

print("\n=== Method 3: Randomized Search ===")
random_search = RandomizedSearchCV(
    KNeighborsClassifier(),
    param_grid_enhanced,
    n_iter=100, 
    scoring='accuracy',
    cv=cv,
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train_corr_scaled, y_train)
best_model_random = random_search.best_estimator_

print(f"Best KNN Params (Random): {random_search.best_params_}")
print(f"Best Random Search Score: {random_search.best_score_:.4f}")

models = {
    'Accuracy Optimized': best_model_acc,
    'F1 Optimized': best_model_f1,
    'Random Search': best_model_random
}

print("\n=== Model Comparison ===")
for name, model in models.items():
    y_pred = model.predict(X_test_corr_scaled)
    y_prob = model.predict_proba(X_test_corr_scaled)[:, 1]
    
    acc, auc, recall, specificity, f1 = evaluate(y_test, y_pred, y_prob)
    
    print(f"\n{name}:")
    print(f" Accuracy: {acc:.4f}")
    print(f" AUC: {auc:.4f}")
    print(f" Recall: {recall:.4f}")
    print(f" Specificity: {specificity:.4f}")
    print(f" F1-Score: {f1:.4f}")

print("\n=== Additional Optimization Strategies ===")

from sklearn.feature_selection import SelectKBest, f_classif
print("1. Feature Selection with SelectKBest:")
selector = SelectKBest(f_classif, k=10) 
X_train_selected = selector.fit_transform(X_train_corr_scaled, y_train)
X_test_selected = selector.transform(X_test_corr_scaled)

knn_selected = KNeighborsClassifier(**grid_acc.best_params_)
knn_selected.fit(X_train_selected, y_train)
y_pred_selected = knn_selected.predict(X_test_selected)
acc_selected = accuracy_score(y_test, y_pred_selected)
print(f" Accuracy with feature selection: {acc_selected:.4f}")

print("\n2. Voting Classifier with multiple KNN models:")
from sklearn.ensemble import VotingClassifier

ensemble = VotingClassifier([
    ('knn_acc', best_model_acc),
    ('knn_f1', best_model_f1),
    ('knn_random', best_model_random)
], voting='soft')

ensemble.fit(X_train_corr_scaled, y_train)
y_pred_ensemble = ensemble.predict(X_test_corr_scaled)
acc_ensemble = accuracy_score(y_test, y_pred_ensemble)
print(f" Ensemble accuracy: {acc_ensemble:.4f}")

print(f"\n3. Dataset size considerations:")
print(f" Training samples: {len(X_train_corr_scaled)}")
print(f" Recommended k range: {int(np.sqrt(len(X_train_corr_scaled))//2)} to {int(np.sqrt(len(X_train_corr_scaled)))}")

print("\n=== FINAL RECOMMENDATION ===")
best_overall = max(models.items(), key=lambda x: accuracy_score(y_test, x[1].predict(X_test_corr_scaled)))
print(f"Best performing model: {best_overall[0]}")
print(f"Best parameters: {best_overall[1].get_params()}")

final_model = best_overall[1]
final_acc = accuracy_score(y_test, final_model.predict(X_test_corr_scaled))
print(f"Final test accuracy: {final_acc:.4f}")

=== Method 1: Grid Search optimizing for Accuracy ===
Fitting 5 folds for each of 84 candidates, totalling 420 fits
Best KNN Params (Accuracy): {'metric': 'minkowski', 'n_neighbors': 13, 'p': 2, 'weights': 'uniform'}
Best CV Accuracy Score: 0.6159

=== Method 2: Grid Search optimizing for F1 Score ===
Fitting 5 folds for each of 84 candidates, totalling 420 fits
Best KNN Params (F1): {'metric': 'minkowski', 'n_neighbors': 3, 'p': 2, 'weights': 'uniform'}
Best CV F1 Score: 0.4276

=== Method 3: Randomized Search ===
Fitting 5 folds for each of 100 candidates, totalling 500 fits
Best KNN Params (Random): {'weights': 'uniform', 'p': 1, 'n_neighbors': 21, 'metric': 'euclidean', 'algorithm': 'auto'}
Best Random Search Score: 0.6190

=== Model Comparison ===

Accuracy Optimized:
  Accuracy: 0.6500
  AUC: 0.6603
  Recall: 0.2619
  Specificity: 0.9310
  F1-Score: 0.3860

F1 Optimized:
  Accuracy: 0.6300
  AUC: 0.6570
  Recall: 0.4286
  Specificity: 0.7759
  F1-Score: 0.4932

Random Search:
  A